# Nishimoto and Gallant (2011) on CRCNS mt-2: Figure 2B with the finer pooling

Paper: Nishimoto S, Gallant JL (2011), J Neurosci 31(41):14551-14564,
doi:10.1523/jneurosci.6801-10.2011. Dataset: CRCNS mt-2 (doi:10.6080/K0DN4374), read from the
public bucket `crcnsarchive/mt-2/live/` (unversioned; `reproduce_nishimoto_2011.ipynb` prints the
MD5 sums of the files). Claims: `claims/nishimoto-2011.yaml`.

The main notebook judges the Figure 2B claims with the regression weights pooled onto 763
regressors (2 x 2 regions). `finer_pooling.ipynb` shows that pooling onto 3 x 3 regions (1465
regressors) raises the compressive and divisive model's mean correlation from 0.448 to 0.518, more
than the differences between the models of Figure 2B. This notebook refits four models of Figure
2B with the 3 x 3 pooling, the linear model without and with divisive normalization and the
compressive model without and with it, on the delay offset each neuron has in the main notebook,
and repeats the comparisons among them. Everything else is as in the main notebook (`mt2.py`,
`pooling_neuron` with `fig2b=True`). The two expansive models and the six luminance-normalized models
were not refitted: on the three smallest neurons each 3 x 3 model took about 15 s of a worker's
time, so the whole set would have taken about an hour, the notebook limit. So
fig2b-expansive-decreases and fig2b-contrast-normalization-no-effect are not repeated here.

| Claim | Here |
|---|---|
| fig2b-compressive-improves | repeated with the finer pooling; judged in the main notebook |
| fig2b-divisive-improves | repeated with the finer pooling; judged in the main notebook |
| fig2b-compressive-divisive-best | repeated with the finer pooling (the four models only); judged in the main notebook |
| fig2b-no-interaction | repeated for the compressive model only; judged in the main notebook |

In [ ]:
import os, time, resource
os.environ.setdefault("OMP_NUM_THREADS", "2")
import numpy as np
import pandas as pd
from concurrent.futures import ProcessPoolExecutor
from scipy.stats import wilcoxon
import mt2

T0 = time.time()
sizes = mt2.list_sizes()
names = sorted(sizes)
N_RUN = int(os.environ.get("MT2_N", len(names)))  # a subset while developing
order = sorted(names, key=lambda n: -sizes[n]) if N_RUN >= len(names) else sorted(names, key=lambda n: sizes[n])[:N_RUN]
t = time.time()
with ProcessPoolExecutor(2) as ex:
    futs = {n: ex.submit(mt2.pooling_neuron, n, False, 2, True) for n in order}
    R = {n: f.result() for n, f in futs.items()}
names_run = [n for n in names if n in R]
print(f"{len(R)} neurons in {(time.time() - t) / 60:.1f} min; per neuron {np.median([R[n]['t_total'] for n in R]):.0f} s median")
print("peak memory of a worker:", resource.getrusage(resource.RUSAGE_CHILDREN).ru_maxrss / 1e6, "GB")

## Data check

In [ ]:
def acc(r):
    return float(np.mean(list(r.values())))

IDX = [n[:6] for n in names_run]
monkey = np.array([R[n]["cellid"][0] for n in names_run])
print("neurons:", len(names_run), "; by monkey:", dict(zip(*np.unique(monkey, return_counts=True))))
print("validation sets:", dict(zip(*np.unique(["+".join(R[n]["sets"]) for n in names_run], return_counts=True))))
print("delay offsets (frames earlier):", dict(zip(*np.unique([R[n]["offset"] for n in names_run], return_counts=True))))
print(f"compressive and divisive model: 2 x 2 pooling mean r {np.mean([acc(R[n]['r2x2']) for n in names_run]):.3f}, "
      f"3 x 3 pooling {np.mean([acc(R[n]['p3x3']['r']) for n in names_run]):.3f} (as in finer_pooling.ipynb)")
print("regressors with 3 x 3 pooling:", R[names_run[0]]["p3x3"]["n_complex"], "complex,", R[names_run[0]]["p3x3"]["n_simple"], "simple")
print("models refitted:", len(R[names_run[0]]["p3x3"]["models"]) + 1)

## The four models

The comparisons are those of the main notebook: the percent change against the simplest model
(linear, no divisive normalization) under readings (a) mean correlation, (b) mean squared
correlation and (c) mean of each neuron's percent change, with Wilcoxon signed-rank tests across
neurons, Bonferroni corrected over 11 comparisons (and over 5, the variant), the counts of the
claims file. A neuron's accuracy is the mean of its two validation correlations where it has both
sets.

In [ ]:
LAB = {(1.0, False): "linear", (0.5, False): "compressive", (1.0, True): "linear +div", (0.5, True): "compressive +div"}
B3 = {k: pd.Series([acc(R[n]["p3x3"]["models"][k]) if k != (0.5, True) else acc(R[n]["p3x3"]["r"]) for n in names_run], index=IDX)
      for k in LAB}
base3 = B3[(1.0, False)]


def change3(k, ref=None):
    a, b = B3[k], base3 if ref is None else B3[ref]
    return (100 * (a.mean() / b.mean() - 1), 100 * ((a**2).mean() / (b**2).mean() - 1),
            100 * np.mean((a - b) / np.abs(b)), wilcoxon(a, b).pvalue)


rows = []
for k, lab in LAB.items():
    pa, pb, pc, p = change3(k) if k != (1.0, False) else (0, 0, 0, np.nan)
    rows.append(dict(model=lab, mean_r=B3[k].mean(), pct_a=pa, pct_b=pb, pct_c=pc, n_better=int((B3[k] > base3).sum()),
                     p=p, p_bonf11=min(1, 11 * p), p_bonf5=min(1, 5 * p)))
print("pooling 3 x 3, every model against the linear model without divisive normalization:")
print(pd.DataFrame(rows).set_index("model").round(4).to_string())


def report3(ks, expect):
    for k, e in zip(ks, expect):
        pa, pb, pc, p = change3(k)
        print(f"pooling 3 x 3, {LAB[k]}: (a) {pa:.1f}%  (b) {pb:.1f}%  (c) {pc:.1f}%  [expected {e} +- 4]; "
              f"better in {(B3[k] > base3).sum()} of {len(IDX)}; p = {p:.3g}, x11 = {min(1, 11 * p):.3g}, x5 = {min(1, 5 * p):.3g}")
        print("   within tolerance, readings (a), (b), (c):", [bool(abs(v - e) <= 4) for v in (pa, pb, pc)])

### fig2b-compressive-improves

> A compressive static output nonlinearity consistently increases predictions over the simplest
> model, by about 16% without and about 19% with divisive normalization.

In [ ]:
report3([(0.5, False), (0.5, True)], [16, 19])

**Result: partially reproduced, as with the 2 x 2 pooling.** With the 3 x 3 pooling the compressive nonlinearity improves predictions in 40 of 45 neurons without and with divisive normalization (corrected p 6.2e-9 and 1.7e-9 over 11). The sizes are 5.6% and 8.0% under reading (a), 11.4% and 16.1% under reading (b), 5.6% and 8.6% under reading (c), against 16 and 19 (+-4); only reading (b) with divisive normalization is within tolerance, as with the 2 x 2 pooling (4.6% and 8.4% under reading (a)).

### fig2b-divisive-improves

> Divisive normalization significantly improves predictions over the simplest model, by about 8%
> with the linear static nonlinearity.

In [ ]:
report3([(1.0, True)], [8])
for mk in np.unique(monkey):
    s = monkey == mk
    print(f"  monkey {mk}: reading (a) {100 * (B3[(1.0, True)][s].mean() / base3[s].mean() - 1):.1f}%")

**Result: not reproduced, as with the 2 x 2 pooling.** With the 3 x 3 pooling divisive normalization with the linear nonlinearity raises the mean correlation by 2.5% (a), 4.7% (b, within 8 +- 4) or 3.1% (c), better in 31 of 45 neurons, Wilcoxon p = 0.0076, corrected p = 0.084 over 11 comparisons (0.038 over 5), so the claim's corrected p < 0.01 is not met. The increase is again larger in monkey v (9.1%, reading (a)) than in monkey f (0.7%).

### fig2b-compressive-divisive-best

> The model with both a compressive nonlinearity and divisive normalization predicts significantly
> better than the model with the compressive nonlinearity alone, and is the simplest model that
> gives accurate predictions.

In [ ]:
cd, c = (0.5, True), (0.5, False)
p = wilcoxon(B3[cd], B3[c]).pvalue
print(f"pooling 3 x 3: compressive + divisive {B3[cd].mean():.3f} against compressive {B3[c].mean():.3f}; better in {(B3[cd] > B3[c]).sum()} of {len(IDX)}")
print(f"Wilcoxon p = {p:.4g}; x11 = {min(1, 11 * p):.4g}; x5 = {min(1, 5 * p):.4g}; x12 = {min(1, 12 * p):.4g}")
print("the other refitted models against compressive + divisive (positive = better than it; the expansive and luminance-normalized models were not refitted):")
for k, lab in LAB.items():
    if k != cd:
        pm = wilcoxon(B3[k], B3[cd]).pvalue
        print(f"  {lab:18s} mean diff {B3[k].mean() - B3[cd].mean():+.4f}  p = {pm:.3g}  x11 = {min(1, 11 * pm):.3g}  x12 = {min(1, 12 * pm):.3g}")

**Result: reproduced, as with the 2 x 2 pooling, among the four models refitted.** The compressive and divisive model (0.518) beats the compressive model (0.506) in 33 of 45 neurons, Wilcoxon p = 0.00075, corrected 0.0083 over 11, 0.0090 over 12 and 0.0038 over 5, each below 0.01 but closer to it than with the 2 x 2 pooling (0.0017 over 11). Neither the linear model nor the linear divisive model is better than it (mean differences -0.039 and -0.026). The expansive and luminance-normalized models were not refitted, so the second condition is checked here on four of the 12 models only.

### fig2b-no-interaction

> The effects of the compressive and expansive nonlinearities do not depend on whether divisive
> normalization is present.

In [ ]:
for div in (False, True):
    for rdg, ref in (("a", (1.0, False)), ("b", (1.0, div))):
        for a_ in (0.5,):  # the expansive models were not refitted
            k = (a_, div)
            pa, pb, pc, p = change3(k, ref)
            print(f"pooling 3 x 3, divisive {div!s:5s} reading ({rdg}) {LAB[k]:17s} against {LAB[ref]:12s}: "
                  f"mean diff {B3[k].mean() - B3[ref].mean():+.4f}, p = {p:.3g}, x11 = {min(1, 11 * p):.3g}")

**Result: reproduced for the compressive model, as with the 2 x 2 pooling.** The compressive model is better than the linear one without divisive normalization (+0.027, corrected p = 6.2e-9) and with it, against the simplest model (reading (a), +0.039, 1.7e-9) and against the linear divisive model (reading (b), +0.026, 6.2e-9). The expansive models were not refitted, so their half of the claim rests on the main notebook only.

## Summary

| Claim | With the 3 x 3 pooling (judged in the main notebook) |
|---|---|
| fig2b-compressive-improves | partially reproduced (5.6% and 8.0%, reading (a); only 16.1% under reading (b) within tolerance) |
| fig2b-divisive-improves | not reproduced (2.5%, reading (a); corrected p = 0.084) |
| fig2b-compressive-divisive-best | reproduced among the four models refitted (corrected p = 0.0083) |
| fig2b-no-interaction | reproduced for the compressive model |

A failure to reproduce here may be a failure of this attempt, not of the paper: see the
reasoning and the differences from the paper's method.

The finer pooling raises every model's mean correlation (linear 0.479, compressive 0.506, linear
divisive 0.492, compressive divisive 0.518) but leaves the outcome of each Figure 2B claim
tested here as it is with the 2 x 2 pooling. The expansive models and the luminance-normalized
models were not refitted. This notebook differs from the paper's method in the same ways as the
main notebook, except for the finer pooling.

In [ ]:
print(f"notebook run time {(time.time() - T0) / 60:.1f} min")